# Stage 6 — GSE270828 perturbation-effect source locator

This notebook inventories public data files without downloading gigabytes of matrices. It will download **only small potential effect-result tables** (up to a size cap) where available, and prepare a compact ZIP with source inventory and diagnostic logs.

**What it does NOT do:** invent differential expression, infer gene targets from guide annotations, or falsely assert an identified file is a differential-expression result. Inspect the output before interpretation.

In [ ]:
import re, io, json, zipfile, csv, urllib.request, urllib.error, html.parser, time
from pathlib import Path
from urllib.parse import urljoin, unquote

OUTPUT=Path('HAR_Stage6_Source_Inventory')
OUTPUT.mkdir(exist_ok=True)
BASE='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/'
ALTERNATES=[BASE,'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/matrix/']
MAX_SMALL_DOWNLOAD_BYTES=15*1024*1024
TIMEOUT_SECONDS=45

def request(url, method='GET'):
    req=urllib.request.Request(url, method=method, headers={'User-Agent':'Mozilla/5.0 (research provenance inventory; GEO public data)','Accept':'*/*'})
    return urllib.request.urlopen(req,timeout=TIMEOUT_SECONDS)

class LinkParser(html.parser.HTMLParser):
    def __init__(self):super().__init__();self.hrefs=[]
    def handle_starttag(self,tag,attrs):
        if tag=='a':
            for k,v in attrs:
                if k=='href' and v:self.hrefs.append(v)

log=[]; found=[]
for site in ALTERNATES:
  try:
    with request(site) as r:
      payload=r.read(1500000)
      status=getattr(r,'status','unknown')
    parser=LinkParser();parser.feed(payload.decode('utf-8','replace'))
    print(site,'status',status,'links',len(parser.hrefs))
    for href in parser.hrefs:
      if href in ('../','./') or href.startswith('?'):continue
      link=urljoin(site,href)
      if not link.startswith(site):continue
      name=unquote(link.rsplit('/',1)[-1])
      if not name or name=='/':continue
      found.append({'directory':site,'filename':name,'url':link})
    log.append({'source':site,'status':'success','n_links':len(parser.hrefs)})
  except Exception as e:
    print('Unable to list:',site,type(e).__name__,str(e)[:160])
    log.append({'source':site,'status':'failed','error':repr(e)[:300]})

found={x['url']:x for x in found}
found=list(found.values())
print('Candidate files in accessible indexes:',len(found))


In [ ]:
# Verify size and optionally fetch small tabular files only.
# This deliberately excludes raw matrix archives / RDS and h5ad even if small.
rows=[]
for entry in found:
 name=entry['filename']; url=entry['url']
 size=None; size_note='unknown'
 try:
    with request(url,'HEAD') as r:
      val=r.headers.get('Content-Length')
      if val is not None:size=int(val);size_note='HEAD content-length'
 except Exception as e:size_note='HEAD unavailable: '+str(e)[:90]
 score=sum(bool(re.search(token,name,re.I)) for token in ['degs','differential','effect','perturb','result','model','screen','table'])
 # no unbounded downloads, no multi-GB matrix files, no RDS/h5ad
 safe_extension=bool(re.search(r'\.(csv|tsv|txt|tab)(\.gz)?$',name,re.I))
 safe_download=bool(safe_extension and size is not None and 0<size<=MAX_SMALL_DOWNLOAD_BYTES and score>0)
 rows.append({**entry,'size_bytes':size,'size_note':size_note,'keyword_matches':score,'eligible_small_download':safe_download})

with (OUTPUT/'GSE270828_Original_File_Inventory.csv').open('w',newline='',encoding='utf8') as f:
 fields=['directory','filename','url','size_bytes','size_note','keyword_matches','eligible_small_download']
 w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)

retrieved=[]
for e in rows:
  if not e['eligible_small_download']:continue
  try:
    with request(e['url']) as r:
      blob=r.read(MAX_SMALL_DOWNLOAD_BYTES+1)
    if len(blob)>MAX_SMALL_DOWNLOAD_BYTES:raise ValueError('Exceeded size cap during GET')
    target=OUTPUT/e['filename']
    target.write_bytes(blob)
    retrieved.append({'filename':e['filename'],'bytes':len(blob),'source_url':e['url']})
    print('Downloaded optional small table:',e['filename'],len(blob))
  except Exception as ex:log.append({'source':e['url'],'status':'small-file-download-failed','error':repr(ex)[:200]})

(OUTPUT/'FETCH_LOG.json').write_text(json.dumps(log,indent=2),encoding='utf8')
(OUTPUT/'SMALL_TABLE_DOWNLOADS.json').write_text(json.dumps(retrieved,indent=2),encoding='utf8')
(OUTPUT/'README.txt').write_text("""This inventory is NOT an effect analysis. Read file names and source pages before interpreting content.

If listing fails, open https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE270828 in a browser and provide a screenshot of the Supplementary files area.

Do not download raw multi-GB RDS/H5AD until an analysis plan is confirmed.
""",encoding='utf8')
zip_path=Path('HAR_Stage6_Source_Inventory.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
  for f in OUTPUT.rglob('*'):
    if f.is_file():z.write(f,arcname=f.name)
print('Ready for upload:',zip_path,'bytes',zip_path.stat().st_size)
try:
 from google.colab import files
 files.download(str(zip_path))
except ImportError:
 print('Outside Colab, retrieve ZIP at:',zip_path.resolve())
